In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from core.brailleimages import read_cleaned

raw_folderpath = "../data/raw/braille-character-dataset/Braille Dataset/Braille Dataset"
processed_folderpath = "../data/processed/braille-character-dataset"

df = read_cleaned(os.path.join(processed_folderpath, 'braille.csv'))

In [137]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

features = df.iloc[:, :-3]
target = df.loc[:, 'letter']

scaler = StandardScaler()
pca = PCA(n_components=.98, random_state=42)
std_pca = Pipeline([('std', scaler), ('pca', pca)])

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=.2, random_state=42)
X_train_ct, X_test_ct, y_train_ct, y_test_ct = train_test_split(df.drop(columns=['letter', 'number']), target, test_size=.2, random_state=42)
# Does recognition work better on non-dimmed images?
mask_light = df.loc[:, 'augmentation'].ne('dim')
df_light = df.loc[mask_light, :]
X_train_light_ct, X_test_light_ct, y_train_light_ct, y_test_light_ct = train_test_split(
    df_light.drop(columns=['letter', 'number']), df_light.loc[:, 'letter'], test_size=.2, random_state=42)


In [139]:
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

forest = RandomForestClassifier(
    max_depth=12, 
    n_estimators=100, 
    class_weight='balanced', 
    max_features='log2', 
    random_state=42)

cat_cols = ['augmentation']

pipe = Pipeline([('std', scaler), ('rf', forest)])
pipe_pca = Pipeline([('std', scaler), ('pca', pca), ('rf', forest)])
pipe_pca_ct = Pipeline([
    ('ct', ColumnTransformer(
        transformers=[
            ('pca', std_pca, X_train.columns[:-1]),
            ('onehot', OneHotEncoder(drop='first'), cat_cols)]
    )),
    ('rf', forest)
])


pipe_pca_ct.fit(X_train_ct, y_train_ct)

preds_train_ct = pipe_pca_ct.predict(X_train_ct)
preds_test_ct = pipe_pca_ct.predict(X_test_ct)
preds_train_light_ct = pipe_pca_ct.predict(X_train_light_ct)
preds_test_light_ct = pipe_pca_ct.predict(X_test_light_ct)
print(accuracy_score(y_train, preds_train_ct))
print(accuracy_score(y_test, preds_test_ct))
print(accuracy_score(y_train_light_ct, preds_train_light_ct))
print(accuracy_score(y_test_light_ct, preds_test_light_ct))

1.0
0.7403846153846154
0.9242788461538461
0.9134615384615384


In [55]:
# tune hyperparameters
# search for max_depth, n_estimators, max_features
from sklearn.model_selection import GridSearchCV, KFold

search_space = {
    'max_depth': [9, 10, 12],
    'min_samples_leaf': [1, 2]
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', max_features='log2', random_state=42), 
    param_grid=search_space, 
    scoring='accuracy', 
    cv=kf,  
    refit=True)
grid_search.fit(features, target)
print(grid_search.best_score_)
print(grid_search.best_params_)
print(grid_search.best_estimator_)

# 0.65
# {'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 1000}
# RandomForestClassifier(class_weight='balanced', max_depth=12,
#                        max_features='log2', n_estimators=1000, random_state=42)

0.6455128205128204
{'max_depth': 12, 'min_samples_leaf': 1}
RandomForestClassifier(class_weight='balanced', max_depth=12,
                       max_features='log2', random_state=42)
